# DownSyndrome

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.DownSyndrome)

class DownSyndrome(LinearReferenceClock):
    pass



In [3]:
model = pya.models.DownSyndrome()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = 'downsyndrome'
model.metadata["data_type"] = 'methylation'
model.metadata["species"] = 'Homo sapiens'
model.metadata["year"] = 2021
model.metadata["approved_by_author"] = '⌛'
model.metadata["citation"] = "Muskens, Ivo S., et al. \"Germline aberrant methylation associated with Down syndrome and its impact on childhood acute lymphoblastic leukemia risk.\" Nature Communications 12.1 (2021): 821."
model.metadata["doi"] = "https://doi.org/10.1038/s41467-021-21064-z"
model.metadata["research_only"] = None
model.metadata["notes"] = None

## Download clock dependencies

In [5]:
os.system(f"curl -sL -o downsyndrome.xlsx https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-021-21064-z/MediaObjects/41467_2021_21064_MOESM6_ESM.xlsx")

0

## Load features

In [6]:
df = pd.read_excel('downsyndrome.xlsx', sheet_name='EWAS_autosomes', skiprows=2)
model.features = df['CpG'].tolist()

/Users/lucascamillo/pyaging/.venv/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## Load weights into base model

In [7]:
weights = torch.tensor(df['beta_overall'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([0.0]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Muskens, Ivo S., et al. "Germline aberrant methylation '
             'associated with Down syndrome and its impact on childhood acute '
             'lymphoblastic leukemia risk." Nature Communications 12.1 (2021): '
             '821.',
 'clock_name': 'downsyndrome',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1038/s41467-021-21064-z',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2021}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg07741821', 'cg02993069', 'cg12477880', 'cg08882472', 'cg24942416', 'cg07841633', 'cg00994804', 'cg11218872', 'cg02451831', 'cg13382072', 'cg24020235', 'cg17239923', 'cg19030331', 'cg03142697', 'cg24999883', 'cg12679760', 'cg23

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[-4.8313],
        [-1.6036],
        [ 4.0192],
        [-1.2427],
        [ 1.9975],
        [-1.7729],
        [ 0.8015],
        [-1.8500],
        [ 1.6546],
        [ 0.3752]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: downsyndrome.xlsx
